**3 Hal yang harus dihasilkan dari praktikum ini :**

1.   reports/data_quality_report.docx — laporan temuan dan keputusan
2.   notebooks/02_cleaning.ipynb — notebook pembersihan yang dapat dijalankan ulang
3. cleaning_log.md — catatan setiap keputusan beserta alasannya

In [115]:
import pandas as pd
import numpy as np
import hashlib, os

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 150)

FILE = 'bedtime_screentime_sleep_debt_kotor.csv'
if not os.path.exists(FILE):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError('Letakkan CSV di folder yang sama dengan notebook ini.')

mentah = pd.read_csv(FILE)
print('Data mentah:', mentah.shape)

Data mentah: (8600, 18)


In [116]:
#Mengamankan data mentah
d = mentah.copy()

jejak = []
def catat(langkah, terdampak, alasan):
    """Setiap perubahan dicatat SAAT dikerjakan, bukan belakangan."""
    jejak.append({'langkah': langkah, 'baris_terdampak': int(terdampak), 'alasan': alasan})
    print(f'  {langkah}: {int(terdampak)} baris')

print('Siap. Data mentah tetap utuh di variabel `mentah`.')


Siap. Data mentah tetap utuh di variabel `mentah`.


**Memastikan Tipe Data**

In [117]:
# Cek bentuk data
d.head(5)

,user_id,age,gender,occupation_type,chronotype,bedtime_phone_minutes,primary_bedtime_app,screen_brightness_pct,blue_light_filter_active,caffeine_post_5pm_mg,physical_activity_min,sleep_latency_min,total_sleep_hours,deep_sleep_pct,rem_sleep_pct,morning_alarm_snoozes,next_day_fatigue_score,sleep_debt_category
0,USR-00001,29,Female,Healthcare / Shift Worker,Intermediate,179,INSTAGRAM / REDDIT,54.0,1,0.0,21,84.4,3.20,23.5,22.2,7,10.0,Severe Sleep Debt
1,USR-00002,58,Female,Student,Intermediate,163,YouTube,76.0,1,92.0,72,86.7,4.06,18.0,21.5,7,10.0,Severe Sleep Debt
2,USR-00003,41,Female,Healthcare / Shift Worker,Night Owl,100,TikTok / Reels,59.0,0,45.0,23,70.5,3.20,18.1,20.5,7,10.0,Severe Sleep Debt
3,USR-00004,36,Non-Binary,Remote Tech,Intermediate,34,YouTube,52.0,0,97.0,49,38.1,6.99,21.6,22.4,1,1.7,Mild Deficit
4,USR-00005,23,Female,Healthcare / Shift Worker,Intermediate,27,TikTok / Reels,82.0,1,0.0,38,32.3,4.58,24.1,NaN,4,5.3,Moderate Debt


In [118]:
# Melakukan penyeragaman data
kolom_num = ['age', 'bedtime_phone_minutes', 'screen_brightness_pct', 'caffeine_post_5pm_mg',
             'physical_activity_min', 'sleep_latency_min', 'total_sleep_hours',
             'deep_sleep_pct', 'rem_sleep_pct', 'morning_alarm_snoozes', 'next_day_fatigue_score']

PLACEHOLDER = {'n/a', 'na', '-', 'nan', 'none', 'null', ''}

# CEK: sel mana yang bukan angka murni?
for k in kolom_num:
    s = d[k].astype('string').str.strip()
    bukan_angka = s.notna() & pd.to_numeric(s, errors='coerce').isna()
    if bukan_angka.any():
        print(f'{k:25s} {bukan_angka.sum():4d} sel teks -> contoh:', s[bukan_angka].unique()[:6])

age                         31 sel teks -> contoh: <StringArray>
['45 tahun', '38 tahun', '28 tahun', '56 tahun', '42 tahun', '22 tahun']
Length: 6, dtype: string
bedtime_phone_minutes       51 sel teks -> contoh: <StringArray>
['41 min', '87 min', '17 min', '64 min', '94 min', '73 min']
Length: 6, dtype: string
rem_sleep_pct               25 sel teks -> contoh: <StringArray>
['-']
Length: 1, dtype: string


In [119]:
# Tindak lanjut penyeragaman
ditangani = {}
for k in kolom_num:
    s = d[k].astype('string').str.strip()
    awal_bukan_angka = (s.notna() & pd.to_numeric(s, errors='coerce').isna()).sum()

    s = s.mask(s.str.lower().isin(PLACEHOLDER))                  # placeholder -> NaN
    s = s.str.extract(r'(-?\d+(?:\.\d+)?)')[0]                   # ambil angkanya saja
    d[k] = pd.to_numeric(s, errors='coerce')                     # jadi float
    ditangani[k] = int(awal_bukan_angka)

catat('Ekstrak angka & placeholder -> NaN', sum(ditangani.values()),
      'Regex ambil angka dari teks bersatuan; N/A, -, NaN dijadikan missing')
print({k: v for k, v in ditangani.items() if v})
print(d[kolom_num].dtypes)

  Ekstrak angka & placeholder -> NaN: 107 baris
{'age': 31, 'bedtime_phone_minutes': 51, 'rem_sleep_pct': 25}
age                         Int64
bedtime_phone_minutes       Int64
screen_brightness_pct     Float64
caffeine_post_5pm_mg      Float64
physical_activity_min       Int64
sleep_latency_min         Float64
total_sleep_hours         Float64
deep_sleep_pct            Float64
rem_sleep_pct             Float64
morning_alarm_snoozes       Int64
next_day_fatigue_score    Float64
dtype: object


In [120]:
# Pengecekan kembali
col = d['bedtime_phone_minutes'].astype(str)
print(col.str.replace(r'[-\d.]+', '', regex=True).str.strip().value_counts())

bedtime_phone_minutes
    8600
Name: count, dtype: int64


In [121]:
# Ubah data dan tipe
d['bedtime_phone_minutes'] = (
    d['bedtime_phone_minutes']
    .astype(str)
    .str.extract(r'(-?\d+\.?\d*)')[0]
    .astype(float)
)

print(d['bedtime_phone_minutes'].dtype)
print(d['bedtime_phone_minutes'].isna().sum(), 'NaN')
print(d['bedtime_phone_minutes'].describe())

float64
0 NaN
count    8600.000000
mean       59.204186
std        36.868346
min         1.000000
25%        31.000000
50%        51.000000
75%        79.000000
max       180.000000
Name: bedtime_phone_minutes, dtype: float64


In [122]:
# Cek type data : apakah sudah benar?
d.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8600 entries, 0 to 8599
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8600 non-null   object 
 1   age                       8600 non-null   Int64  
 2   gender                    8600 non-null   object 
 3   occupation_type           8429 non-null   object 
 4   chronotype                8473 non-null   object 
 5   bedtime_phone_minutes     8600 non-null   float64
 6   primary_bedtime_app       8600 non-null   object 
 7   screen_brightness_pct     8429 non-null   Float64
 8   blue_light_filter_active  8600 non-null   object 
 9   caffeine_post_5pm_mg      8429 non-null   Float64
 10  physical_activity_min     8600 non-null   Int64  
 11  sleep_latency_min         8344 non-null   Float64
 12  total_sleep_hours         8600 non-null   Float64
 13  deep_sleep_pct            8343 non-null   Float64
 14  rem_slee

## Teks, Placeholder, & **kategori**

In [123]:
# CONTOH 4.1 — buang spasi pada SELURUH kolom teks sekaligus
kolom_teks = [c for c in d.columns if d[c].dtype == object or str(d[c].dtype) == 'str']

n_spasi = 0
for kol in kolom_teks:
    s = d[kol].astype('string')
    n_spasi += int((s != s.str.strip()).sum())
    d[kol] = s.str.strip()

catat('Buang spasi awal dan akhir kolom teks', n_spasi,
      'Spasi tersembunyi memecah groupby dan melambungkan nunique')

  Buang spasi awal dan akhir kolom teks: 500 baris


In [124]:
# Melakukan pengecekan variasi nilai
kolom_kat = ['gender', 'occupation_type', 'chronotype', 'primary_bedtime_app']

def kunci(s):
    """Bentuk normal untuk membandingkan: tanpa spasi tepi, huruf kecil, pemisah seragam."""
    return (s.astype('string').str.strip().str.lower()
             .str.replace(r'[\s_\-]+', ' ', regex=True))

for k in kolom_kat:
    print(f'\n=== {k} ===')
    print('Nilai unik mentah :', d[k].nunique(dropna=True))
    print('Nilai unik ternormalisasi:', kunci(d[k]).nunique(dropna=True))
    print('Missing:', d[k].isna().sum())

    # Tampilkan varian mentah yang dikelompokkan per bentuk normalnya
    varian = (d[[k]].assign(_kunci=kunci(d[k]))
                    .dropna()
                    .groupby('_kunci')[k]
                    .agg(lambda x: sorted(x.unique().tolist())))
    print(varian.to_string())


=== gender ===
Nilai unik mentah : 12
Nilai unik ternormalisasi: 5
Missing: 0
_kunci
f                                                          [F]
female                                [FEMALE, Female, female]
m                                                          [M]
male                                        [MALE, Male, male]
non binary    [NON-BINARY, Non Binary, Non-Binary, non-binary]

=== occupation_type ===
Nilai unik mentah : 15
Nilai unik ternormalisasi: 5
Missing: 171
_kunci
corporate 9 to 5             [CORPORATE 9-TO-5, Corporate 9-to-5, corporate...
freelance / creative         [FREELANCE / CREATIVE, Freelance / Creative, f...
healthcare / shift worker    [HEALTHCARE / SHIFT WORKER, Healthcare / Shift...
remote tech                            [REMOTE TECH, Remote Tech, remote tech]
student                                            [STUDENT, Student, student]

=== chronotype ===
Nilai unik mentah : 8
Nilai unik ternormalisasi: 3
Missing: 127
_kunci
intermediate    

In [125]:
# Penyeragaman konsistensi penulisan pada gender, occupation-type, chronotype, primary_bedtime_app
LABEL_BAKU = {
    'gender'             : ['Female', 'Male', 'Non-Binary'],
    'occupation_type'    : ['Corporate 9-to-5', 'Student', 'Remote Tech',
                            'Freelance / Creative', 'Healthcare / Shift Worker'],
    'chronotype'         : ['Intermediate', 'Night Owl', 'Morning Lark'],
    'primary_bedtime_app': ['News / Reading', 'Streaming (Netflix/Hulu)', 'Messaging / Chat',
                            'YouTube', 'Instagram / Reddit', 'TikTok / Reels'],
}

# Diputuskan manual (kunci ditulis dalam bentuk ternormalisasi)
PETA_MANUAL = {
    'gender': {'f': 'Female', 'm': 'Male'},
    # kolom lain tidak butuh keputusan manual
}

for k, valid in LABEL_BAKU.items():
    sebelum = d[k].nunique()                       # hitung SEBELUM transformasi
    kunci_k = kunci(d[k])

    # Otomatis: kunci ternormalisasi -> label baku. Manual: ditambahkan di atasnya.
    peta = {kunci(pd.Series([v]))[0]: v for v in valid}
    peta.update(PETA_MANUAL.get(k, {}))

    hasil = kunci_k.map(peta)

    # Pengaman: ada nilai non-missing yang tidak terpetakan?
    tak_terpetakan = set(kunci_k[kunci_k.notna() & hasil.isna()].unique())
    assert not tak_terpetakan, f'{k}: belum terpetakan -> {tak_terpetakan}'

    d[k] = hasil
    catat(f'Seragamkan {k}', sebelum - d[k].nunique(),
          'Spasi/huruf/pemisah oleh kode; singkatan diputuskan manual di PETA_MANUAL')
    print(k, '->', sorted(d[k].dropna().unique()))

  Seragamkan gender: 9 baris
gender -> ['Female', 'Male', 'Non-Binary']
  Seragamkan occupation_type: 10 baris
occupation_type -> ['Corporate 9-to-5', 'Freelance / Creative', 'Healthcare / Shift Worker', 'Remote Tech', 'Student']
  Seragamkan chronotype: 5 baris
chronotype -> ['Intermediate', 'Morning Lark', 'Night Owl']
  Seragamkan primary_bedtime_app: 12 baris
primary_bedtime_app -> ['Instagram / Reddit', 'Messaging / Chat', 'News / Reading', 'Streaming (Netflix/Hulu)', 'TikTok / Reels', 'YouTube']


In [126]:
# Penyeragaman kategori, dengan peta keputusan yang eksplisit
KATEGORI_VALID = {'Optimal Recovery', 'Mild Deficit', 'Moderate Debt', 'Severe Sleep Debt'}

# Hitung SEBELUM transformasi apa pun, agar log mencerminkan seluruh langkah
sebelum = d['sleep_debt_category'].nunique()

# Otomatis oleh kode: spasi dan huruf besar/kecil
d['sleep_debt_category'] = d['sleep_debt_category'].astype('string').str.strip().str.title()

# Diputuskan manual: salah ketik. Kunci ditulis dalam bentuk SETELAH title(),
# agar benar-benar cocok.
PETA_sleep_debt_category = {
    'Sever Sleep Debt': 'Severe Sleep Debt',
    'Moderat Debt'    : 'Moderate Debt',
    'Mild Defisit'    : 'Mild Deficit',
    'Optimal Recovry' : 'Optimal Recovery',
}
d['sleep_debt_category'] = d['sleep_debt_category'].replace(PETA_sleep_debt_category)

# Pengaman: pastikan tidak ada nilai di luar 4 kategori valid
tak_terpetakan = set(d['sleep_debt_category'].dropna().unique()) - KATEGORI_VALID
assert not tak_terpetakan, f'Masih ada kategori tak terpetakan: {tak_terpetakan}'

catat('Seragamkan sleep_debt_category', sebelum - d['sleep_debt_category'].nunique(),
      'Huruf dan spasi oleh kode; 4 salah ketik diputuskan manual lalu dicatat di peta')

print(sorted(d['sleep_debt_category'].dropna().unique()))

  Seragamkan sleep_debt_category: 4 baris
['Mild Deficit', 'Moderate Debt', 'Optimal Recovery', 'Severe Sleep Debt']


In [127]:
# Hasil Penyeragaman kategori
p5_kat      = d['sleep_debt_category']

print('P5_kat', p5_kat)


P5_kat 0       Severe Sleep Debt
1       Severe Sleep Debt
2       Severe Sleep Debt
3            Mild Deficit
4           Moderate Debt
              ...        
8595        Moderate Debt
8596    Severe Sleep Debt
8597         Mild Deficit
8598    Severe Sleep Debt
8599        Moderate Debt
Name: sleep_debt_category, Length: 8600, dtype: string


In [128]:
# Penyeragaman yes true, no false pada blue_light_filter_active
# CEK
print(d['blue_light_filter_active'].astype('string').str.strip().str.lower().value_counts(dropna=False))

# BERSIHKAN
sebelum = d['blue_light_filter_active'].astype('string').nunique()
PETA_BINER = {'1': 1, 'yes': 1, 'true': 1,
              '0': 0, 'no': 0, 'false': 0}

kunci_b = d['blue_light_filter_active'].astype('string').str.strip().str.lower()
hasil = kunci_b.map(PETA_BINER)

tak_terpetakan = set(kunci_b[kunci_b.notna() & hasil.isna()].unique())
assert not tak_terpetakan, f'Belum terpetakan: {tak_terpetakan}'

d['blue_light_filter_active'] = hasil.astype('Int64')   # Int64 agar NaN tetap boleh
catat('Seragamkan blue_light_filter_active', sebelum - d['blue_light_filter_active'].nunique(),
      'Yes/No/True/False/1/0 dipetakan ke 1/0')
print(d['blue_light_filter_active'].value_counts(dropna=False))

blue_light_filter_active
0        4215
1        3697
no        250
yes       200
true      125
false     113
Name: count, dtype: Int64
  Seragamkan blue_light_filter_active: 6 baris
blue_light_filter_active
0    4578
1    4022
Name: count, dtype: Int64


### DUPLIKAT

In [129]:
# duplikat persis
n_persis = int(d.duplicated().sum())
d = d.drop_duplicates()
catat('Hapus duplikat persis', n_persis, 'Seluruh kolom identik; satu baris disimpan')

  Hapus duplikat persis: 100 baris


In [130]:
# kunci utama yang kosong
n_kosong = int(d['user_id'].isna().sum())
d = d[d['user_id'].notna()]
catat('Buang baris tanpa user_id', n_kosong,
      'Kunci utama kosong tidak dapat ditelusuri kembali ke sumber')

  Buang baris tanpa user_id: 0 baris


## NILAI MUSTAHIL

In [131]:
rekap = {}
for k, (lo, hi) in ATURAN.items():
    m = (d[k] < lo) | (d[k] > hi)
    rekap[k] = int(m.sum())
    if m.any():
        print(f'{k:25s} {m.sum():3d} melanggar [{lo},{hi}] -> {sorted(d.loc[m, k].unique())}')
    d.loc[m, k] = np.nan

total = sum(rekap.values())
catat('Nilai di luar batas domain -> NaN', total, 'Aturan domain per kolom, bukan IQR')

if total == 0:
    print('PERINGATAN: 0 pelanggaran. Cek apakah sel ini sudah pernah dijalankan '
          'atau data sudah dibersihkan sebelumnya.')

age                         8 melanggar [18,65] -> [np.int64(-5), np.int64(0), np.int64(999)]
total_sleep_hours           8 melanggar [2,12] -> [np.float64(-3.0), np.float64(0.0), np.float64(25.0)]
screen_brightness_pct       6 melanggar [0,100] -> [np.float64(-20.0), np.float64(130.0), np.float64(150.0)]
caffeine_post_5pm_mg        4 melanggar [0,600] -> [np.float64(-50.0), np.float64(2500.0)]
sleep_latency_min           4 melanggar [0,300] -> [np.float64(-10.0), np.float64(999.0)]
  Nilai di luar batas domain -> NaN: 30 baris


### **Nilai kosong, dan mengapa tidak ada pilihan yang netral**

In [132]:
d.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8500 entries, 0 to 8599
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8500 non-null   string 
 1   age                       8492 non-null   Int64  
 2   gender                    8500 non-null   object 
 3   occupation_type           8330 non-null   object 
 4   chronotype                8373 non-null   object 
 5   bedtime_phone_minutes     8500 non-null   float64
 6   primary_bedtime_app       8500 non-null   object 
 7   screen_brightness_pct     8324 non-null   Float64
 8   blue_light_filter_active  8500 non-null   Int64  
 9   caffeine_post_5pm_mg      8326 non-null   Float64
 10  physical_activity_min     8500 non-null   Int64  
 11  sleep_latency_min         8241 non-null   Float64
 12  total_sleep_hours         8492 non-null   Float64
 13  deep_sleep_pct            8245 non-null   Float64
 14  rem_sleep_pct

In [133]:
# CEK: rekap missing sebelum imputasi
miss = d.isna().sum()
miss_pct = (miss / len(d) * 100).round(2)
print(pd.DataFrame({'missing': miss, '%': miss_pct})[miss > 0])

                       missing     %
age                          8  0.09
occupation_type            170  2.00
chronotype                 127  1.49
screen_brightness_pct      176  2.07
caffeine_post_5pm_mg       174  2.05
sleep_latency_min          259  3.05
total_sleep_hours            8  0.09
deep_sleep_pct             255  3.00
rem_sleep_pct               60  0.71


In [134]:
# BERSIHKAN
# a) Kategori: label 'Unknown', bukan modus, agar tidak mengarang kategori
for k in ['occupation_type', 'chronotype']:
    n = d[k].isna().sum()
    d[k] = d[k].fillna('Unknown')
    catat(f'Imputasi {k}', int(n), "Diisi label 'Unknown'")

# b) Numerik umum: median global
for k in ['age', 'screen_brightness_pct', 'caffeine_post_5pm_mg']:
    n = d[k].isna().sum()
    d[k] = d[k].fillna(d[k].median())
    catat(f'Imputasi {k}', int(n), 'Median global')

# c) Numerik fisiologis: median per sleep_debt_category
for k in ['total_sleep_hours', 'sleep_latency_min', 'deep_sleep_pct', 'rem_sleep_pct']:
    n = d[k].isna().sum()
    med_grup = d.groupby('sleep_debt_category')[k].transform('median')
    d[k] = d[k].fillna(med_grup).fillna(d[k].median())   # cadangan jika grupnya kosong
    catat(f'Imputasi {k}', int(n), 'Median per sleep_debt_category')

# d) Rapikan tipe untuk kolom bulat
for k in ['age', 'screen_brightness_pct', 'caffeine_post_5pm_mg', 'bedtime_phone_minutes']:
    d[k] = d[k].round().astype(int)

# PENGAMAN AKHIR
assert d.isna().sum().sum() == 0, d.isna().sum()[lambda s: s > 0]
print(d.dtypes)

  Imputasi occupation_type: 170 baris
  Imputasi chronotype: 127 baris
  Imputasi age: 8 baris
  Imputasi screen_brightness_pct: 176 baris
  Imputasi caffeine_post_5pm_mg: 174 baris
  Imputasi total_sleep_hours: 8 baris
  Imputasi sleep_latency_min: 259 baris
  Imputasi deep_sleep_pct: 255 baris
  Imputasi rem_sleep_pct: 60 baris
user_id                     string[python]
age                                  int64
gender                              object
occupation_type                     object
chronotype                          object
bedtime_phone_minutes                int64
primary_bedtime_app                 object
screen_brightness_pct                int64
blue_light_filter_active             Int64
caffeine_post_5pm_mg                 int64
physical_activity_min                Int64
sleep_latency_min                  Float64
total_sleep_hours                  Float64
deep_sleep_pct                     Float64
rem_sleep_pct                      Float64
morning_alarm_snoozes  

In [135]:
d.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8500 entries, 0 to 8599
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8500 non-null   string 
 1   age                       8500 non-null   int64  
 2   gender                    8500 non-null   object 
 3   occupation_type           8500 non-null   object 
 4   chronotype                8500 non-null   object 
 5   bedtime_phone_minutes     8500 non-null   int64  
 6   primary_bedtime_app       8500 non-null   object 
 7   screen_brightness_pct     8500 non-null   int64  
 8   blue_light_filter_active  8500 non-null   Int64  
 9   caffeine_post_5pm_mg      8500 non-null   int64  
 10  physical_activity_min     8500 non-null   Int64  
 11  sleep_latency_min         8500 non-null   Float64
 12  total_sleep_hours         8500 non-null   Float64
 13  deep_sleep_pct            8500 non-null   Float64
 14  rem_sleep_pct

### HASIL PEMBERSIHAN

In [136]:
d = d.reset_index(drop=True)

p5_baris_bersih  = len(d)
p5_dibuang       = len(mentah) - len(d)
p5_sleep_latency_min_kosong = int(d['sleep_latency_min'].isna().sum())

print('P5_baris_bersih', p5_baris_bersih)
print('P5_dibuang', p5_dibuang)
print('P5_sleep_latency_min_kosong', p5_sleep_latency_min_kosong)

P5_baris_bersih 8500
P5_dibuang 100
P5_sleep_latency_min_kosong 0


**MENYUSUN ULANG BENTUK TABEL**

In [137]:
# pivot: dari panjang ke lebar (untuk dibaca manusia)
lebar = (d.pivot_table(index='occupation_type', columns='sleep_debt_category', values='total_sleep_hours',
                       aggfunc='sum', observed=True)
          .fillna(0).round(0).astype(int))
lebar

sleep_debt_category,Mild Deficit,Moderate Debt,Optimal Recovery,Severe Sleep Debt
occupation_type,,,,
Corporate 9-to-5,5094,7786,4314,687
Freelance / Creative,1649,2506,1214,244
Healthcare / Shift Worker,611,3573,131,615
Remote Tech,3788,6040,3261,475
Student,3010,4384,2432,374
Unknown,331,484,223,37


In [138]:
# merge: menggabungkan tabel lain
# Tabel ringkasan per anggota, lalu digabungkan kembali ke total tidur.
profil_anggota = (d.groupby('user_id')
                    .agg(jumlah_=('bedtime_phone_minutes', 'sum'),
                         total_tidur=('total_sleep_hours', 'sum'))
                    .reset_index())

gabung = d.merge(profil_anggota, on='user_id', how='left', validate='many_to_one')
print('Sebelum:', d.shape, '-> sesudah:', gabung.shape)
profil_anggota.head(3)

Sebelum: (8500, 18) -> sesudah: (8500, 20)


,user_id,jumlah_,total_tidur
0,USR-00001,179,3.2
1,USR-00002,163,4.06
2,USR-00003,100,3.2


## **Pipeline yang dapat dijalankan**

In [139]:
# CONTOH 10.1 — kerangka pipeline
def bersihkan(df_mentah):
    """Terima data mentah, kembalikan data bersih beserta jejak keputusannya."""
    x = df_mentah.copy()
    log = []
    def c(langkah, n, alasan):
        log.append({'langkah': langkah, 'baris_terdampak': int(n), 'alasan': alasan})

    # 1 tipe data
    # 2 spasi
    # 3 placeholder
    # 4 kategori
    # 5 duplikat
    # 6 nilai mustahil
    # 7 nilai kosong
    # 8 kolom turunan

    return x.reset_index(drop=True), pd.DataFrame(log)

print('Salin langkah-langkah di atas ke dalam fungsi ini untuk notebook M2 Anda.')

Salin langkah-langkah di atas ke dalam fungsi ini untuk notebook M2 Anda.


In [140]:
# CONTOH 10.2 — simpan hasil
os.makedirs('data/processed', exist_ok=True)
os.makedirs('reports', exist_ok=True)

d.to_csv('data/processed/bedtime_screen_sleep_debt_bersih.csv', index=False)
print('Tersimpan:', d.shape)
print()
print('Data mentah tidak disentuh:', mentah.shape)

Tersimpan: (8500, 18)

Data mentah tidak disentuh: (8600, 18)


### **Cleaning Log**

In [141]:
# cetak log
log = pd.DataFrame(jejak)
log.index = range(1, len(log) + 1)
log

,langkah,baris_terdampak,alasan
1,Ekstrak angka & placeholder -> NaN,107,"Regex ambil angka dari teks bersatuan; N/A, -,..."
2,Buang spasi awal dan akhir kolom teks,500,Spasi tersembunyi memecah groupby dan melambun...
3,Seragamkan gender,9,Spasi/huruf/pemisah oleh kode; singkatan diput...
4,Seragamkan occupation_type,10,Spasi/huruf/pemisah oleh kode; singkatan diput...
5,Seragamkan chronotype,5,Spasi/huruf/pemisah oleh kode; singkatan diput...
6,Seragamkan primary_bedtime_app,12,Spasi/huruf/pemisah oleh kode; singkatan diput...
7,Seragamkan sleep_debt_category,4,Huruf dan spasi oleh kode; 4 salah ketik diput...
8,Seragamkan blue_light_filter_active,6,Yes/No/True/False/1/0 dipetakan ke 1/0
9,Hapus duplikat persis,100,Seluruh kolom identik; satu baris disimpan
10,Buang baris tanpa user_id,0,Kunci utama kosong tidak dapat ditelusuri kemb...


In [114]:
# 1) Catat perubahan bedtime_phone_minutes ke log (hitung dari data mentah)
langkah_bp = "Konversi bedtime_phone_minutes ke numerik (buang satuan 'min')"

if langkah_bp not in log['langkah'].values:
    col = mentah['bedtime_phone_minutes']
    bp = pd.to_numeric(col, errors='coerce')
    n_terdampak = int((bp.isna() & col.notna()).sum())

    log.loc[log.index.max() + 1, ['langkah', 'baris_terdampak', 'alasan']] = [
        langkah_bp,
        n_terdampak,
        "Kolom bertipe teks karena sebagian nilai berformat '<angka> min'; "
        "diekstrak angkanya dengan regex dan diubah ke float agar bisa dihitung"
    ]

# 2) Simpan sebagai Markdown untuk repositori
baris = ['# Cleaning Log', '',
         'Dataset: bedtime_screen_sleep_debt.csv  ',
         f'Baris mentah: {len(mentah)} -> baris bersih: {len(d)}  ',
         '', '| # | Langkah | Baris terdampak | Alasan |', '|---|---|---|---|']
for i, r in log.iterrows():
    baris.append(f"| {i} | {r['langkah']} | {r['baris_terdampak']} | {r['alasan']} |")

with open('cleaning_log.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(baris))

print('\n'.join(baris[-5:]))   # tampilkan baris terakhir supaya langkah baru terlihat

| 17 | Imputasi total_sleep_hours | 8.0 | Median per sleep_debt_category |
| 18 | Imputasi sleep_latency_min | 259.0 | Median per sleep_debt_category |
| 19 | Imputasi deep_sleep_pct | 255.0 | Median per sleep_debt_category |
| 20 | Imputasi rem_sleep_pct | 60.0 | Median per sleep_debt_category |
| 21 | Konversi bedtime_phone_minutes ke numerik (buang satuan 'min') | 51.0 | Kolom bertipe teks karena sebagian nilai berformat '<angka> min'; diekstrak angkanya dengan regex dan diubah ke float agar bisa dihitung |


### **KELENGKAPAN LOG**

In [142]:
# jumlah langkah di cleaning log

p5_langkah   = len(jejak)
print('P5_langkah =', p5_langkah)

P5_langkah = 20


## **Verifikasi**

In [143]:
def diagnosa_singkat(data, nama):
    print(f'=== {nama} ===')
    print('  Baris                     :', len(data))
    print('  Duplikat persis           :', int(data.duplicated().sum()))
    print('  Kategori unik             :', data['occupation_type'].astype(str).nunique())
    print('  id_anggota unik           :', data['user_id'].astype(str).nunique())
    print('  total_sleep_hours numerik :', pd.api.types.is_numeric_dtype(data['total_sleep_hours']))

    # Konversi aman: teks non-angka -> NaN
    bp = pd.to_numeric(data['bedtime_phone_minutes'], errors='coerce')
    print('  bedtime_phone_minutes tipe asli         :', data['bedtime_phone_minutes'].dtype)
    print('  bedtime_phone_minutes non-angka (jadi NaN):', int(bp.isna().sum() - data['bedtime_phone_minutes'].isna().sum()))
    print('  bedtime_phone_minutes <= 0              :', int((bp <= 0).sum()))
    print()

diagnosa_singkat(mentah, 'SEBELUM')
diagnosa_singkat(d, 'SESUDAH')

=== SEBELUM ===
  Baris                     : 8600
  Duplikat persis           : 100
  Kategori unik             : 26
  id_anggota unik           : 8500
  total_sleep_hours numerik : True
  bedtime_phone_minutes tipe asli         : object
  bedtime_phone_minutes non-angka (jadi NaN): 51
  bedtime_phone_minutes <= 0              : 0

=== SESUDAH ===
  Baris                     : 8500
  Duplikat persis           : 0
  Kategori unik             : 6
  id_anggota unik           : 8500
  total_sleep_hours numerik : True
  bedtime_phone_minutes tipe asli         : int64
  bedtime_phone_minutes non-angka (jadi NaN): 0
  bedtime_phone_minutes <= 0              : 0



In [144]:
d.head(5)

,user_id,age,gender,occupation_type,chronotype,bedtime_phone_minutes,primary_bedtime_app,screen_brightness_pct,blue_light_filter_active,caffeine_post_5pm_mg,physical_activity_min,sleep_latency_min,total_sleep_hours,deep_sleep_pct,rem_sleep_pct,morning_alarm_snoozes,next_day_fatigue_score,sleep_debt_category
0,USR-00001,29,Female,Healthcare / Shift Worker,Intermediate,179,Instagram / Reddit,54,1,0,21,84.4,3.2,23.5,22.2,7,10.0,Severe Sleep Debt
1,USR-00002,58,Female,Student,Intermediate,163,YouTube,76,1,92,72,86.7,4.06,18.0,21.5,7,10.0,Severe Sleep Debt
2,USR-00003,41,Female,Healthcare / Shift Worker,Night Owl,100,TikTok / Reels,59,0,45,23,70.5,3.2,18.1,20.5,7,10.0,Severe Sleep Debt
3,USR-00004,36,Non-Binary,Remote Tech,Intermediate,34,YouTube,52,0,97,49,38.1,6.99,21.6,22.4,1,1.7,Mild Deficit
4,USR-00005,23,Female,Healthcare / Shift Worker,Intermediate,27,TikTok / Reels,82,1,0,38,32.3,4.58,24.1,19.2,4,5.3,Moderate Debt


In [145]:
d.info()
d.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8500 entries, 0 to 8499
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8500 non-null   string 
 1   age                       8500 non-null   int64  
 2   gender                    8500 non-null   object 
 3   occupation_type           8500 non-null   object 
 4   chronotype                8500 non-null   object 
 5   bedtime_phone_minutes     8500 non-null   int64  
 6   primary_bedtime_app       8500 non-null   object 
 7   screen_brightness_pct     8500 non-null   int64  
 8   blue_light_filter_active  8500 non-null   Int64  
 9   caffeine_post_5pm_mg      8500 non-null   int64  
 10  physical_activity_min     8500 non-null   Int64  
 11  sleep_latency_min         8500 non-null   Float64
 12  total_sleep_hours         8500 non-null   Float64
 13  deep_sleep_pct            8500 non-null   Float64
 14  rem_slee

(8500, 18)